In [1]:
"""
Unified Rigorous Comprehensive Atlas Test Suite

Tests EVERY element of the tree structure against exact expectations for the 'sample_files' project.
Includes specific stress tests for Type Inference, Scope Management (via notes),
Navigation Variants, and FQN Variants.
Each test validates specific nodes, attributes, types, and notes exist with correct values.
Fails immediately on first mismatch with detailed error message.

Excludes: Visualization, Serialization, Error Handling for malformed code.
"""

import sys
import ast  # Required for node type checks in inference tests
from io import StringIO
from analyzer import build_complete_atlas
from analyzer import (
    ProjectNode, PackageNode, ModuleNode, ClassNode, FunctionNode,
    ArgumentNode, ReturnNode, InstanceAttributeNode, ClassAttributeNode,
    StateNode, ImportNode, ImportFromNode, TypeNode, StateContainerNode
)
from analyzer.core.navigation import TraversalScope
from analyzer.notes import (
    MissingArgumentTypeHint, MissingReturnTypeHint,
    MissingClassAttributeTypeHint, MissingInstanceAttributeTypeHint,
    UnsupportedExpressionType, IncorrectTypeAnnotation,
    ScopeAddition, BaseClassResolution, TypeInference, ParameterDiscovery, TypeInferenceFailure
)

# --- Helper Functions ---

def assert_eq(actual, expected, description):
    """Assert equality with detailed error message."""
    assert actual == expected, f"❌ {description}\n  Expected: {repr(expected)}\n  Got: {repr(actual)}"

def assert_node_exists(node, fqn_desc):
    """Assert node exists."""
    assert node is not None, f"❌ Node not found: {fqn_desc}"

def assert_node_type(node, expected_type, fqn_desc):
    """Assert node is correct type."""
    assert isinstance(node, expected_type), \
        f"❌ Wrong node type for {fqn_desc}\n  Expected: {expected_type.__name__}\n  Got: {type(node).__name__}"

def assert_has_note(node, note_type, description):
    """Assert node has at least one note of the specified type."""
    notes = node.get_notes(note_type)
    assert len(notes) >= 1, f"❌ {description}: Expected note {note_type.__name__} but none found on {node}"
    # Return notes for further inspection if needed
    return notes

def assert_has_exactly_one_note(node, note_type, description):
    """Assert node has exactly one note of the specified type."""
    notes = node.get_notes(note_type)
    assert len(notes) == 1, f"❌ {description}: Expected exactly one {note_type.__name__} note, found {len(notes)} on {node}"
    return notes[0]

def get_type_inference_note(node, var_name):
    """Find the TypeInference note for a specific variable on a node."""
    notes = node.get_notes(TypeInference)
    for note in notes:
        if hasattr(note, 'variable_name') and note.variable_name == var_name: # Check if attr exists
            return note
    return None

# --- Main Test Function ---

def run_atlas_stress_test():
    """Executes the unified Atlas stress test suite."""
    print("="*80)
    print("UNIFIED RIGOROUS COMPREHENSIVE ATLAS TEST")
    print("="*80)
    print("\nBuilding and analyzing sample files...")

    # Capture stdout to verify silent operation
    captured = StringIO()
    old_stdout = sys.stdout
    sys.stdout = captured

    # Build and analyze the project
    try:
        project = build_complete_atlas('sample_files')
        project.analyze()
    except Exception as e:
        sys.stdout = old_stdout # Restore stdout before error
        print(f"\n❌ FAILED during project build/analysis: {e}")
        raise # Re-raise the exception
    finally:
        sys.stdout = old_stdout # Ensure stdout is always restored

    output = captured.getvalue()

    assert not output.strip(), f"❌ Expected silent operation but got output:\n{output}"
    print("✓ Silent operation confirmed")

    # --- TEST 1: PROJECT STRUCTURE ---
    print("\n" + "="*80)
    print("TEST 1: PROJECT STRUCTURE")
    print("="*80)

    assert_node_type(project, ProjectNode, "root")
    assert_eq(project.name, "sample_files", "Project name")
    assert_eq(project.fqn, "sample_files", "Project FQN")
    print("✓ Project node correct")

    modules = project.list_all_modules()
    module_fqns = {m.fqn for m in modules}
    expected_modules = {
        "sample_files.root_module",
        "sample_files.subpackage.nested_module"
    }
    assert_eq(module_fqns, expected_modules, "Module FQNs")
    print(f"✓ Found exactly {len(expected_modules)} expected modules")

    packages = project.list_packages()
    package_fqns = {p.fqn for p in packages}
    expected_packages = {"sample_files.subpackage"}
    assert_eq(package_fqns, expected_packages, "Package FQNs")
    print(f"✓ Found exactly {len(expected_packages)} expected packages")

    # --- TEST 2: ROOT_MODULE - MODULE STATE ---
    print("\n" + "="*80)
    print("TEST 2: ROOT_MODULE - MODULE STATE")
    print("="*80)

    root_module = project.get_node_by_fqn("sample_files.root_module")
    assert_node_exists(root_module, "sample_files.root_module")
    assert_node_type(root_module, ModuleNode, "sample_files.root_module")
    print("✓ root_module exists and is ModuleNode")

    state_containers = root_module._state_containers
    assert_eq(len(state_containers), 4, "Number of state containers in root_module")

    all_state_vars = []
    for container in state_containers:
        # Ensure _state_variables exists before extending
        if hasattr(container, '_state_variables'):
             all_state_vars.extend(container._state_variables)

    assert_eq(len(all_state_vars), 4, "Number of state variables in root_module")
    print(f"✓ Found {len(all_state_vars)} state variables in {len(state_containers)} containers")

    state_names = {s.name for s in all_state_vars if hasattr(s, 'name')} # Check name exists
    expected_state_names = {"VERSION", "MAX_ITEMS", "debug_mode", "default_timeout"}
    assert_eq(state_names, expected_state_names, "State variable names")
    print(f"✓ All expected state variables exist: {sorted(list(state_names))}") # Use list() for sorting

    # --- TEST 3: ROOT_MODULE - BASEENTITY CLASS ---
    print("\n" + "="*80)
    print("TEST 3: ROOT_MODULE - BASEENTITY CLASS")
    print("="*80)

    base_entity = project.get_node_by_fqn("sample_files.root_module.BaseEntity")
    assert_node_exists(base_entity, "sample_files.root_module.BaseEntity")
    assert_node_type(base_entity, ClassNode, "sample_files.root_module.BaseEntity")
    print("✓ BaseEntity class correct")

    init_method = base_entity.dot("__init__")
    assert_node_exists(init_method, "BaseEntity.__init__")
    print("✓ BaseEntity.__init__ method exists")

    init_args = init_method.list_arguments() # Use navigation method
    assert_eq(len(init_args), 3, "Number of __init__ arguments")
    arg_names = [arg.name for arg in init_args]
    assert_eq(arg_names, ["self", "entity_id", "name"], "__init__ argument names")
    print(f"✓ __init__ has correct arguments: {arg_names}")

    param_notes = init_method.get_notes(ParameterDiscovery)
    param_note_names = {note.parameter_name for note in param_notes if hasattr(note, 'parameter_name')} # Check attr exists
    assert "entity_id" in param_note_names, "__init__ should have ParameterDiscovery note for entity_id"
    assert "name" in param_note_names, "__init__ should have ParameterDiscovery note for name"
    print("✓ __init__ has expected ParameterDiscovery notes")

    entity_id_arg = init_method.dot("entity_id")
    assert entity_id_arg._type is not None, "entity_id should have type annotation"
    print("✓ entity_id argument correct with type annotation")
    name_arg = init_method.dot("name")
    assert name_arg._type is not None, "name should have type annotation"
    print("✓ name argument correct with type annotation")

    instance_attrs = base_entity.list_instance_attributes() # Use navigation method
    assert_eq(len(instance_attrs), 3, "Number of BaseEntity instance attributes")
    entity_id_attr = base_entity.dot("entity_id")
    assert entity_id_attr._type is not None, "entity_id attribute should have type"
    print("✓ entity_id instance attribute correct with type")
    name_attr = base_entity.dot("name")
    assert name_attr._type is not None, "name attribute should have type"
    print("✓ name instance attribute correct with type")
    created_at_attr = base_entity.dot("created_at")
    assert created_at_attr._type is None, "created_at should NOT have type"
    assert_has_exactly_one_note(created_at_attr, MissingInstanceAttributeTypeHint, "created_at attribute")
    print("✓ created_at instance attribute correct (untyped, has violation note)")

    get_id = base_entity.dot("get_id")
    get_id_return = get_id.dot("return")
    assert get_id_return._type is not None, "get_id should have return type"
    print("✓ get_id method correct with return type")
    validate = base_entity.dot("validate")
    validate_return = validate.dot("return")
    assert validate_return._type is None, "validate should NOT have return type"
    assert_has_exactly_one_note(validate_return, MissingReturnTypeHint, "validate return")
    print("✓ validate method correct (missing return type, has violation note)")

    # --- TEST 4: ROOT_MODULE - CONFIG CLASS ---
    print("\n" + "="*80)
    print("TEST 4: ROOT_MODULE - CONFIG CLASS")
    print("="*80)

    config = project.get_node_by_fqn("sample_files.root_module.Config")
    assert_node_exists(config, "sample_files.root_module.Config")
    assert_node_type(config, ClassNode, "Config")
    print("✓ Config class exists")

    class_attrs = config.list_class_attributes() # Use navigation method
    assert_eq(len(class_attrs), 2, "Number of Config class attributes")
    max_conn = config.dot("MAX_CONNECTIONS")
    assert max_conn._type is not None, "MAX_CONNECTIONS should have type"
    print("✓ MAX_CONNECTIONS class attribute correct with type")
    default_host = config.dot("DEFAULT_HOST")
    assert default_host._type is None, "DEFAULT_HOST should NOT have type"
    assert_has_exactly_one_note(default_host, MissingClassAttributeTypeHint, "DEFAULT_HOST attribute")
    print("✓ DEFAULT_HOST class attribute correct (untyped, has violation note)")

    # --- TEST 5: ROOT_MODULE - FUNCTIONS ---
    print("\n" + "="*80)
    print("TEST 5: ROOT_MODULE - FUNCTIONS")
    print("="*80)

    calc_total = project.get_node_by_fqn("sample_files.root_module.calculate_total")
    assert_node_exists(calc_total, "calculate_total")
    assert_node_type(calc_total, FunctionNode, "calculate_total")
    items_arg = calc_total.dot("items")
    assert items_arg._type is not None, "items should have type annotation"
    print("✓ calculate_total function correct with typed arguments")

    format_name = project.get_node_by_fqn("sample_files.root_module.format_name")
    assert_node_exists(format_name, "format_name")
    first_arg = format_name.dot("first")
    assert first_arg._type is None, "first should NOT have type"
    assert_has_exactly_one_note(first_arg, MissingArgumentTypeHint, "first argument")
    last_arg = format_name.dot("last")
    assert last_arg._type is None, "last should NOT have type"
    assert_has_exactly_one_note(last_arg, MissingArgumentTypeHint, "last argument")
    format_return = format_name.dot("return")
    assert format_return._type is None, "format_name should NOT have return type"
    assert_has_exactly_one_note(format_return, MissingReturnTypeHint, "format_name return")
    print("✓ format_name function correct (untyped, has violation notes)")

    # --- TEST 6: NESTED_MODULE - PRODUCT CLASS (INHERITANCE) ---
    print("\n" + "="*80)
    print("TEST 6: NESTED_MODULE - PRODUCT CLASS (INHERITANCE)")
    print("="*80)

    product = project.get_node_by_fqn("sample_files.subpackage.nested_module.Product")
    assert_node_exists(product, "Product")
    assert_node_type(product, ClassNode, "Product")
    print("✓ Product class exists")

    assert_eq(len(product.base_classes), 1, "Product base class count") # Use property
    assert_eq(product.base_classes[0], "BaseEntity", "Product base class name")
    print("✓ Product declares BaseEntity as base class")

    base_res_note = assert_has_exactly_one_note(product, BaseClassResolution, "Product BaseClassResolution")
    assert_eq(base_res_note.base_name, "BaseEntity", "BaseClassResolution note name")
    assert "BaseEntity" in base_res_note.base_fqn, "BaseClassResolution note FQN"
    print(f"✓ Product base_class_fqns resolved: {product.base_class_fqns} (validated by note)")

    inherited_name = product.dot("name")
    assert_node_exists(inherited_name, "Product inherited name attribute")
    print("✓ Product.dot('name') correctly finds inherited attribute from BaseEntity")

    product_attrs = product.list_instance_attributes() # Use navigation method
    assert_eq(len(product_attrs), 4, "Product instance attribute count")
    price_attr = product.dot("price")
    assert price_attr._type is not None
    print("✓ price attribute correct")
    tags_attr = product.dot("tags")
    assert tags_attr._type is not None
    print("✓ tags attribute correct")
    metadata_attr = product.dot("metadata")
    assert metadata_attr._type is not None
    print("✓ metadata attribute correct")
    in_stock_attr = product.dot("in_stock")
    assert in_stock_attr._type is None
    assert_has_exactly_one_note(in_stock_attr, MissingInstanceAttributeTypeHint, "in_stock attribute")
    print("✓ in_stock attribute correct (untyped, has violation note)")

    product_methods = product.list_methods() # Use navigation method
    assert_eq(len(product_methods), 4, "Product method count")
    add_tag = product.dot("add_tag")
    tag_arg = add_tag.dot("tag")
    assert tag_arg._type is None
    add_tag_return = add_tag.dot("return")
    assert add_tag_return._type is None
    print("✓ add_tag method correct (untyped, violations expected)")

    # --- TEST 7: NESTED_MODULE - INVENTORY & STORE CLASSES ---
    print("\n" + "="*80)
    print("TEST 7: NESTED_MODULE - INVENTORY & STORE CLASSES")
    print("="*80)

    inventory = project.get_node_by_fqn("sample_files.subpackage.nested_module.Inventory")
    assert_node_exists(inventory, "Inventory")
    items_attr = inventory.dot("items")
    assert items_attr._type is not None, "items should have type"
    print("✓ Inventory.items correct")
    count_attr = inventory.dot("count")
    assert count_attr._type is None, "count should NOT have type"
    print("✓ Inventory.count correct (untyped)")

    store = project.get_node_by_fqn("sample_files.subpackage.nested_module.Store")
    assert_node_exists(store, "Store")
    inventory_attr = store.dot("inventory")
    assert inventory_attr._type is not None, "inventory should have type"
    print("✓ Store.inventory correct")
    is_open_attr = store.dot("is_open")
    assert is_open_attr._type is None, "is_open should NOT have type"
    print("✓ Store.is_open correct (untyped)")

    # --- TEST 8: IMPORT HANDLING ---
    print("\n" + "="*80)
    print("TEST 8: IMPORT HANDLING")
    print("="*80)

    root_imports = root_module.list_imports() # Use navigation method
    assert len(root_imports) > 0, "root_module should have imports"
    import_count = len(root_imports)
    print(f"✓ root_module has {import_count} import statements")
    scope_notes = root_module.get_notes(ScopeAddition)
    import_scope_notes = [n for n in scope_notes if hasattr(n, 'entity_type') and n.entity_type == 'import'] # Check attr exists
    sys_note = next((n for n in import_scope_notes if hasattr(n, 'entity_name') and n.entity_name == 'sys'), None) # Check attr exists
    assert sys_note is not None and sys_note.entity_fqn == 'sys', "ScopeAddition note for 'import sys'"
    datetime_note = next((n for n in import_scope_notes if hasattr(n, 'entity_name') and n.entity_name == 'datetime'), None) # Check attr exists
    assert datetime_note is not None and datetime_note.entity_fqn == 'datetime.datetime', "ScopeAddition note for 'from datetime import datetime'"
    print("✓ root_module has expected ScopeAddition notes for imports")

    nested_module = project.get_node_by_fqn("sample_files.subpackage.nested_module")
    nested_imports = nested_module.list_imports() # Use navigation method
    assert len(nested_imports) > 0, "nested_module should have imports"
    print(f"✓ nested_module has {len(nested_imports)} import statements")
    scope_notes_nested = nested_module.get_notes(ScopeAddition)
    base_entity_import_note = next((n for n in scope_notes_nested if hasattr(n, 'entity_name') and n.entity_name == 'BaseEntity'), None) # Check attr exists
    assert base_entity_import_note is not None, "ScopeAddition note for 'from root_module import BaseEntity'"
    assert base_entity_import_note.entity_fqn.endswith('root_module.BaseEntity'), \
        f"BaseEntity import FQN incorrect: {base_entity_import_note.entity_fqn}"
    print("✓ nested_module has expected ScopeAddition notes for imports (incl. relative)")

    # --- TEST 9: TYPE INFERENCE ENGINE STRESS TEST ---
    print("\n" + "="*80)
    print("TEST 9: TYPE INFERENCE ENGINE STRESS TEST")
    print("="*80)

    note = get_type_inference_note(nested_module, "count")
    assert note and note.inferred_type == "int", "Type inference for 'count = 42'"
    print("✓ Inferred literal 'int'")
    note = get_type_inference_note(nested_module, "name")
    assert note and note.inferred_type == "str", "Type inference for 'name = \"TestProduct\"'"
    print("✓ Inferred literal 'str'")
    note = get_type_inference_note(nested_module, "is_valid")
    assert note and note.inferred_type == "bool", "Type inference for 'is_valid = True'"
    print("✓ Inferred literal 'bool'")

    note = get_type_inference_note(nested_module, "product")
    assert note and note.inferred_type == "sample_files.subpackage.nested_module.Product", \
           f"Type inference for 'product = Product(...)' - got {note.inferred_type if note else 'None'}"
    print("✓ Inferred constructor call 'Product'")
    note = get_type_inference_note(nested_module, "store")
    assert note and note.inferred_type == "sample_files.subpackage.nested_module.Store", \
           f"Type inference for 'store = Store(...)' - got {note.inferred_type if note else 'None'}"
    print("✓ Inferred constructor call 'Store'")

    products_state_node = nested_module.dot("products")
    if products_state_node:
        products_type_node = products_state_node.dot("type")
        if products_type_node:
             assert products_type_node.type_string == "List[Product]", "Type string for products annotation"
             print("✓ Container annotation 'List[Product]' correctly parsed")

    note = get_type_inference_note(nested_module, "product_name")
    if note:
        assert note.inferred_type == "str", \
            f"Type inference for 'product.name' (inherited) - got {note.inferred_type}"
        print("✓ Inferred inherited attribute access 'product.name' -> str")
    else:
         fail_note = nested_module.get_notes(TypeInferenceFailure)
         assert any(hasattr(n, 'variable_name') and n.variable_name == "product_name" for n in fail_note), \
             "Expected TypeInferenceFailure or successful inference for product_name" # Check attr exists
         print("✓ Type inference failed as expected (or succeeded) for inherited attribute 'product.name'")

    note = get_type_inference_note(nested_module, "product_price")
    if note:
        assert note.inferred_type == "Decimal", \
            f"Type inference for 'product.price' - got {note.inferred_type}"
        print("✓ Inferred direct attribute access 'product.price' -> Decimal")
    else:
         fail_note = nested_module.get_notes(TypeInferenceFailure)
         assert any(hasattr(n, 'variable_name') and n.variable_name == "product_price" for n in fail_note), \
                "Expected TypeInferenceFailure or successful inference for product_price" # Check attr exists
         print("✓ Type inference failed as expected (or succeeded) for direct attribute 'product.price'")

    note = get_type_inference_note(nested_module, "product_id")
    if note:
        assert note.inferred_type == "str", \
            f"Type inference for 'product.get_id()' - got {note.inferred_type}"
        print("✓ Inferred method call 'product.get_id()' -> str")
    else:
         fail_note = nested_module.get_notes(TypeInferenceFailure)
         assert any(hasattr(n, 'variable_name') and n.variable_name == "product_id" for n in fail_note), \
             "Expected TypeInferenceFailure or successful inference for product_id" # Check attr exists
         print("✓ Type inference failed as expected (or succeeded) for method call 'product.get_id()'")

    note = get_type_inference_note(nested_module, "discount_price")
    assert note is None, "Type inference for 'product.calculate_discount()' should fail (missing return type)"
    fail_notes = nested_module.get_notes(TypeInferenceFailure)
    assert any(hasattr(n, 'variable_name') and n.variable_name == "discount_price" for n in fail_notes), \
        "Expected TypeInferenceFailure note for 'discount_price'" # Check attr exists
    print("✓ Inference correctly failed for method call with missing return type ('discount_price')")

    note = get_type_inference_note(nested_module, "first_product")
    if note:
        assert note.inferred_type == "sample_files.subpackage.nested_module.Product", \
            f"Type inference for 'products[0]' - got {note.inferred_type}"
        print("✓ Inferred subscript 'products[0]' -> Product")
    else:
         fail_note = nested_module.get_notes(TypeInferenceFailure)
         assert any(hasattr(n, 'variable_name') and n.variable_name == "first_product" for n in fail_note), \
                "Expected TypeInferenceFailure or successful inference for first_product" # Check attr exists
         print("✓ Type inference failed as expected (or succeeded) for subscript 'products[0]'")

    note = get_type_inference_note(nested_module, "lookup_product")
    if note:
        assert note.inferred_type == "sample_files.subpackage.nested_module.Product", \
            f"Type inference for 'product_dict[\"p1\"]' - got {note.inferred_type}"
        print("✓ Inferred subscript 'product_dict[\"p1\"]' -> Product")
    else:
         fail_note = nested_module.get_notes(TypeInferenceFailure)
         assert any(hasattr(n, 'variable_name') and n.variable_name == "lookup_product" for n in fail_note), \
                "Expected TypeInferenceFailure or successful inference for lookup_product" # Check attr exists
         print("✓ Type inference failed as expected (or succeeded) for subscript 'product_dict[\"p1\"]'")

    wrong_type_notes = nested_module.get_notes(IncorrectTypeAnnotation)
    wrong_type_note = next((n for n in wrong_type_notes if hasattr(n, 'annotation') and n.annotation == "int" and hasattr(n, 'inferred') and n.inferred == "str"), None) # Check attrs exist
    assert wrong_type_note is not None, \
        "Expected IncorrectTypeAnnotation note for 'wrong_type: int = \"not an int\"'"
    assert_eq(wrong_type_note.annotation, "int", "IncorrectTypeAnnotation annotation for wrong_type")
    assert_eq(wrong_type_note.inferred, "str", "IncorrectTypeAnnotation inferred type for wrong_type")
    print("✓ Detected IncorrectTypeAnnotation for 'wrong_type: int = \"not an int\"'")

    # --- TEST 10: UNSUPPORTED EXPRESSIONS ---
    print("\n" + "="*80)
    print("TEST 10: UNSUPPORTED EXPRESSION NOTES")
    print("="*80)

    unsupported_notes = nested_module.get_notes(UnsupportedExpressionType)
    assert len(unsupported_notes) >= 4, "Should have at least 4 UnsupportedExpressionType notes for BinOp, Compare, IfExp, JoinedStr"
    expression_types = {n.expression_type for n in unsupported_notes if hasattr(n, 'expression_type')} # Check attr exists
    expected_types = {"BinOp", "Compare", "IfExp", "JoinedStr"}
    assert expected_types.issubset(expression_types), \
        f"Should have unsupported notes for {expected_types}, got {expression_types}"
    print(f"✓ Found {len(unsupported_notes)} UnsupportedExpressionType notes")
    print(f"  Expression types: {sorted(list(expression_types))}") # Use list() for sorting

    # --- TEST 11: NAVIGATION VARIANTS STRESS TEST ---
    print("\n" + "="*80)
    print("TEST 11: NAVIGATION VARIANTS STRESS TEST")
    print("="*80)

    pkgs_direct = project.list_packages()
    assert_eq(len(pkgs_direct), 1, "project.list_packages() count")
    assert_eq(pkgs_direct[0].name, "subpackage", "project.list_packages() name")
    print("✓ project.list_packages() (CONTEXT->DIRECT) correct")

    pkgs_child = project.list_child_packages()
    assert_eq(len(pkgs_child), 1, "project.list_child_packages() count")
    assert_eq(pkgs_child[0].name, "subpackage", "project.list_child_packages() name")
    print("✓ project.list_child_packages() (DIRECT) correct")

    pkgs_all = project.list_all_packages()
    assert_eq(len(pkgs_all), 1, "project.list_all_packages() count")
    assert_eq(pkgs_all[0].name, "subpackage", "project.list_all_packages() name")
    print("✓ project.list_all_packages() (CASCADE) correct")

    mods_direct = project.list_modules()
    assert_eq(len(mods_direct), 1, "project.list_modules() count")
    assert_eq(mods_direct[0].name, "root_module", "project.list_modules() name")
    print("✓ project.list_modules() (CONTEXT->DIRECT) correct")

    mods_all = project.list_all_modules()
    mod_all_names = {m.name for m in mods_all}
    assert_eq(len(mods_all), 2, "project.list_all_modules() count")
    assert_eq(mod_all_names, {"root_module", "nested_module"}, "project.list_all_modules() names")
    print("✓ project.list_all_modules() (CASCADE) correct")

    classes_all = project.list_classes()
    class_all_names = {c.name for c in classes_all}
    expected_classes = {"BaseEntity", "Config", "Product", "Inventory", "Store"}
    assert_eq(len(classes_all), len(expected_classes), "project.list_classes() count")
    assert_eq(class_all_names, expected_classes, "project.list_classes() names")
    print("✓ project.list_classes() (CONTEXT->CASCADE) correct")

    be_methods_context = base_entity.list_methods()
    be_method_names_context = {m.name for m in be_methods_context}
    assert_eq(be_method_names_context, {"__init__", "get_id", "validate"}, "BaseEntity.list_methods() names")
    print("✓ BaseEntity.list_methods() (CONTEXT->CASCADE) correct")

    be_methods_direct = base_entity.list_child_methods()
    be_method_names_direct = {m.name for m in be_methods_direct}
    assert_eq(be_method_names_direct, {"__init__", "get_id", "validate"}, "BaseEntity.list_child_methods() names")
    print("✓ BaseEntity.list_child_methods() (DIRECT) correct")

    be_methods_all = base_entity.list_all_methods()
    be_method_names_all = {m.name for m in be_methods_all}
    assert_eq(be_method_names_all, {"__init__", "get_id", "validate"}, "BaseEntity.list_all_methods() names")
    print("✓ BaseEntity.list_all_methods() (CASCADE) correct")

    init_args_context = init_method.list_arguments()
    init_arg_names_context = {a.name for a in init_args_context}
    assert_eq(init_arg_names_context, {"self", "entity_id", "name"}, "BaseEntity.__init__.list_arguments() names")
    print("✓ BaseEntity.__init__.list_arguments() (CONTEXT->DIRECT) correct")

    init_args_direct = init_method.list_child_arguments()
    init_arg_names_direct = {a.name for a in init_args_direct}
    assert_eq(init_arg_names_direct, {"self", "entity_id", "name"}, "BaseEntity.__init__.list_child_arguments() names")
    print("✓ BaseEntity.__init__.list_child_arguments() (DIRECT) correct")

    # --- TEST 12: FQN VARIANTS STRESS TEST ---
    print("\n" + "="*80)
    print("TEST 12: FQN VARIANTS STRESS TEST")
    print("="*80)

    assert_eq(project.fqn, "sample_files", "Project FQN")
    assert_eq(project.xfqn, "Project(sample_files)", "Project XFQN")
    assert_eq(project.cfqn, "Project(sample_files)", "Project CFQN")
    print("✓ Project FQN variants correct")

    subpackage = project.get_node_by_fqn("sample_files.subpackage")
    assert_node_exists(subpackage, "subpackage")
    assert_eq(subpackage.fqn, "sample_files.subpackage", "Package FQN")
    assert_eq(subpackage.xfqn, "Project(sample_files).Package(subpackage)", "Package XFQN")
    assert_eq(subpackage.cfqn, "Project(sample_files).Package(subpackage)", "Package CFQN")
    print("✓ Package FQN variants correct")

    assert_eq(nested_module.fqn, "sample_files.subpackage.nested_module", "Module FQN")
    assert_eq(nested_module.xfqn, "Project(sample_files).Package(subpackage).Module(nested_module)", "Module XFQN")
    assert_eq(nested_module.cfqn, "Project(sample_files).Package(subpackage).Module(nested_module)", "Module CFQN")
    print("✓ Module FQN variants correct")

    assert_eq(base_entity.fqn, "sample_files.root_module.BaseEntity", "Class FQN")
    assert_eq(base_entity.xfqn, "Project(sample_files).Module(root_module).Class(BaseEntity)", "Class XFQN")
    assert_eq(base_entity.cfqn, "Project(sample_files).Module(root_module).Class(BaseEntity)", "Class CFQN")
    print("✓ Class FQN variants correct")

    assert_eq(init_method.fqn, "sample_files.root_module.BaseEntity.__init__", "Method FQN")
    assert_eq(init_method.xfqn, "Project(sample_files).Module(root_module).Class(BaseEntity).Function(__init__)", "Method XFQN")
    assert_eq(init_method.cfqn, "Project(sample_files).Module(root_module).Class(BaseEntity).Function(__init__)", "Method CFQN")
    print("✓ Method FQN variants correct")

    assert_eq(entity_id_arg.fqn, "sample_files.root_module.BaseEntity.__init__.entity_id", "Argument FQN")
    assert_eq(entity_id_arg.xfqn, "Project(sample_files).Module(root_module).Class(BaseEntity).Function(__init__).Argument(entity_id)", "Argument XFQN")
    assert_eq(entity_id_arg.cfqn, "Project(sample_files).Module(root_module).Class(BaseEntity).Function(__init__).Argument(entity_id)", "Argument CFQN")
    print("✓ Argument FQN variants correct")

    assert_eq(name_attr.fqn, "sample_files.root_module.BaseEntity.name", "Instance Attribute FQN")
    assert_eq(name_attr.xfqn, "Project(sample_files).Module(root_module).Class(BaseEntity).InstanceAttribute(name)", "Instance Attribute XFQN")
    assert_eq(name_attr.cfqn, "Project(sample_files).Module(root_module).Class(BaseEntity).InstanceAttribute(name)", "Instance Attribute CFQN")
    print("✓ Instance Attribute FQN variants correct")

    assert_eq(max_conn.fqn, "sample_files.root_module.Config.MAX_CONNECTIONS", "Class Attribute FQN")
    assert_eq(max_conn.xfqn, "Project(sample_files).Module(root_module).Class(Config).ClassAttribute(MAX_CONNECTIONS)", "Class Attribute XFQN")
    assert_eq(max_conn.cfqn, "Project(sample_files).Module(root_module).Class(Config).ClassAttribute(MAX_CONNECTIONS)", "Class Attribute CFQN")
    print("✓ Class Attribute FQN variants correct")

    version_state = project.get_node_by_fqn("sample_files.root_module.VERSION")
    assert_node_exists(version_state, "VERSION state")
    assert_eq(version_state.fqn, "sample_files.root_module.VERSION", "State FQN")
    # Check CFQN structure carefully, startswith might be too lenient if path changes
    expected_cfqn_state = "Project(sample_files).Module(root_module).StateContainer().State(VERSION)"
    assert_eq(version_state.cfqn, expected_cfqn_state, "State CFQN")
    #assert version_state.cfqn.startswith("Project(sample_files).Module(root_module).StateContainer().State(VERSION)"), "State CFQN"
    print("✓ State FQN variants correct (incl. Container in CFQN)")

    state_container = version_state.parent
    assert_node_type(state_container, StateContainerNode, "State Container")
    assert_eq(state_container.fqn, "sample_files.root_module", "Container FQN (pass-through)")
    assert_eq(state_container.xfqn, "Project(sample_files).Module(root_module)", "Container XFQN")
    assert_eq(state_container.cfqn, "Project(sample_files).Module(root_module).StateContainer()", "Container CFQN")
    print("✓ Container FQN variants correct (pass-through FQN, typed XFQN/CFQN)")

    # --- FINAL SUMMARY ---
    print("\n" + "="*80)
    print("ENHANCED TEST SUITE COMPLETE - ALL TESTS PASSED!")
    print("="*80)
    print("\nValidated:")
    print("  ✓ Project structure")
    print("  ✓ Classes, Attributes (Class/Instance), Methods")
    print("  ✓ Functions, Arguments, Returns, Module State")
    print("  ✓ Type annotations (present and missing)")
    print("  ✓ Inheritance resolution and navigation")
    print("  ✓ All violation notes (missing type hints)")
    print("  ✓ All analysis notes (ScopeAddition, BaseClassResolution, TypeInference, ParameterDiscovery)")
    print("  ✓ Specific TypeInference results for literals, constructors, attributes, methods, subscripts")
    print("  ✓ TypeInferenceFailure notes for expected failures")
    print("  ✓ IncorrectTypeAnnotation note")
    print("  ✓ All limitation notes (UnsupportedExpressionType)")
    print("  ✓ Import handling and ScopeAddition notes")
    print("  ✓ Navigation Variants (list_*, list_child_*, list_all_*)")
    print("  ✓ FQN Variants (.fqn, .xfqn, .cfqn) for various node types")
    print("\nAtlas core features are more rigorously validated! 🎉")


# --- Execute the Test ---
if __name__ == "__main__":
    try:
        run_atlas_stress_test()
    except AssertionError as e:
        print(f"\n\n💥 TEST FAILED: {e}")
        sys.exit(1)
    except Exception as e:
        print(f"\n\n💥 UNEXPECTED ERROR DURING TEST: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

UNIFIED RIGOROUS COMPREHENSIVE ATLAS TEST

Building and analyzing sample files...
✓ Silent operation confirmed

TEST 1: PROJECT STRUCTURE
✓ Project node correct
✓ Found exactly 2 expected modules
✓ Found exactly 1 expected packages

TEST 2: ROOT_MODULE - MODULE STATE
✓ root_module exists and is ModuleNode
✓ Found 4 state variables in 4 containers
✓ All expected state variables exist: ['MAX_ITEMS', 'VERSION', 'debug_mode', 'default_timeout']

TEST 3: ROOT_MODULE - BASEENTITY CLASS
✓ BaseEntity class correct
✓ BaseEntity.__init__ method exists
✓ __init__ has correct arguments: ['self', 'entity_id', 'name']
✓ __init__ has expected ParameterDiscovery notes
✓ entity_id argument correct with type annotation
✓ name argument correct with type annotation
✓ entity_id instance attribute correct with type
✓ name instance attribute correct with type
✓ created_at instance attribute correct (untyped, has violation note)
✓ get_id method correct with return type
✓ validate method correct (missing retur

In [2]:
from analyzer import build_complete_atlas
project = build_complete_atlas('sample_files')
project.analyze()
print('✓ Smoke test passed')

✓ Smoke test passed


In [3]:
from analyzer import build_complete_atlas
project = build_complete_atlas('sample_files')
project.analyze()

# Verify type inference still works
module = project.list_modules()[0]
from analyzer.notes import TypeInference
notes = module.get_notes(TypeInference)
print(f'✓ Found {len(notes)} type inference notes')

✓ Found 8 type inference notes
